In [1]:
#!/usr/bin/env python3
"""
Train NepBPE on a large text corpus, with phase timing and vocab saving.

Build the Rust extension first (from the crate root):
    maturin develop --release
    #   or:  pip install -e .    (if you use setuptools-rust)

Then:
    python train.py
"""

import time
import json
import sys
from tiny_llm_scratch_with_tokenizer import PyNepBPETokenizer

# ----------------------------------------------------------------------------
# Config
# ----------------------------------------------------------------------------
DATA_PATH      = "wiki_bank_other.txt"
VOCAB_BUDGET   = 48_000      # total vocab (base + merges). 32k-64k is reasonable.
THETA          = 100         # V_ambiguous frequency gate
MIN_WORD_FREQ  = 3           # drop word types seen < 3x (cuts the hapax tail; big win at this scale)
PROGRESS_LINES = 500_000     # build-phase heartbeat every N lines (0 = silent)
PROGRESS_MERGES = 1_000      # train-phase heartbeat every N merges (0 = silent)
OUT_VOCAB      = "nepbpe_vocab_new.tsv"

# Orthographic folding rules (pattern -> replacement), longest-match-first.
# These are string->string so multi-codepoint folds work. Extend per your policy.
FOLDING_RULES = [
    ("सङ्ग", "संग"),   # explicit nasal conjunct -> anusvara
    ("सँग", "संग"),    # chandrabindu -> anusvara
]

# Base vocabulary
DEVANAGARI = [chr(c) for c in range(0x0900, 0x0980)]     # the 128 U+0900..U+097F codepoints
PUNCTUATION = list(".,!?;:()[]{}\"'`-–—…/\\@#%&*+=<>|~") + ["।", "॥"]  # danda + double danda
SEED_MORPHEMES: list[str] = []   # e.g. ["हरू", "मा", "को", "लाई", ...] once you curate them
V_STRICT: list[str]       = []   # unconditionally frozen closed-class morphemes
V_AMBIGUOUS: list[str]    = []   # frequency-gated morphemes


def main() -> None:
    tok = PyNepBPETokenizer(folding_rules=FOLDING_RULES)

    base = tok.initialize_vocab(
        DEVANAGARI,
        SEED_MORPHEMES,
        PUNCTUATION,
        V_STRICT,
        V_AMBIGUOUS,
    )
    print(f"base vocab initialized: {base} tokens", flush=True)


    print(f"training on {DATA_PATH} (budget={VOCAB_BUDGET}, min_word_freq={MIN_WORD_FREQ}) ...",
          flush=True)

    t0 = time.perf_counter()
    final = tok.train_from_file(
        DATA_PATH,
        VOCAB_BUDGET,
        THETA,
        MIN_WORD_FREQ,
        PROGRESS_LINES,
        PROGRESS_MERGES,
    )
    wall = time.perf_counter() - t0

    print(f"\n=== training complete ===", flush=True)
    print(f"final vocab size : {final}", flush=True)
    print(f"wall clock       : {wall:.1f}s  ({wall/60:.1f} min)", flush=True)

    # Save the vocab so the run isn't lost. (id -> surface, tab-separated.)
    n = tok.vocab_size()
    with open(OUT_VOCAB, "w", encoding="utf-8") as f:
        for i in range(n):
            surface = tok.get_token_surface(i)
            # escape tab/newline in surface so the TSV stays parseable
            surface = surface.replace("\\", "\\\\").replace("\t", "\\t").replace("\n", "\\n")
            f.write(f"{i}\t{surface}\n")
    print(f"vocab written    : {OUT_VOCAB}", flush=True)

    # Tiny sanity check: fertility on a couple of sample lines.
    samples = ["नेपालको इतिहास धेरै पुरानो छ", "म विद्यालय जान्छु"]
    for s in samples:
        ids = tok.encode(s)
        ok = tok.decode(ids) == tok.normalize(s)
        print(f"  '{s}' -> {len(ids)} tokens | roundtrip={'OK' if ok else 'FAIL'}", flush=True)


if __name__ == "__main__":
    sys.exit(main())

base vocab initialized: 388 tokens
training on wiki_bank_other.txt (budget=48000, min_word_freq=3) ...


[build] 500000 lines | 71027124 word-occ | 1860030 unique | 58.1s
[build] 1000000 lines | 92599667 word-occ | 2230321 unique | 75.0s
[build] 1500000 lines | 113950025 word-occ | 2519677 unique | 91.9s
[build] 2000000 lines | 135470101 word-occ | 2757161 unique | 108.7s
[build] 2500000 lines | 157055955 word-occ | 2995716 unique | 125.6s
[build] 3000000 lines | 175031484 word-occ | 3184512 unique | 139.7s
[build] 3500000 lines | 184643303 word-occ | 3293517 unique | 147.4s
[build] 4000000 lines | 194074494 word-occ | 3390729 unique | 155.0s
[build] 4500000 lines | 203297537 word-occ | 3479381 unique | 162.5s
[build] 5000000 lines | 212607554 word-occ | 3567380 unique | 170.1s
[build] 5500000 lines | 221939546 word-occ | 3652201 unique | 177.7s
[build] 6000000 lines | 231297634 word-occ | 3731682 unique | 186.4s
[build] 6500000 lines | 240509540 word-occ | 3810051 unique | 194.3s
[build] 7000000 lines | 249783538 word-occ | 3884249 unique | 202.2s
[build] 7500000 lines | 259125077 word-o


=== training complete ===
final vocab size : 48000
wall clock       : 2476.5s  (41.3 min)
vocab written    : nepbpe_vocab_new.tsv
  'नेपालको इतिहास धेरै पुरानो छ' -> 9 tokens | roundtrip=OK
  'म विद्यालय जान्छु' -> 5 tokens | roundtrip=OK


SystemExit: 

/home/lang-chain/Documents/Astra_agentic_RAG/.venv/lib/python3.11/site-packages/IPython/core/interactiveshell.py:3709: UserWarning: To exit: use 'exit', 'quit', or Ctrl-D.
  warn("To exit: use 'exit', 'quit', or Ctrl-D.", stacklevel=1)
